# Рекомендательные системы в PySpark

В современной индустрии (кинотеатры, маркетплейсы, музыкальные стриминги) рекомендательные системы обычно строятся по двухэтапной схеме:
1. **Отбор кандидатов (Candidates Generation):** из миллионов товаров быстро выбираются несколько сотен потенциально интересных. Здесь используются быстрые эвристики, контентный поиск или векторные индексы.
2. **Ранжирование (Ranking):** выбранные кандидаты оцениваются тяжелой моделью машинного обучения, которая предсказывает точную оценку или вероятность клика и сортирует их для пользователя.

Для примера возьмём очищенные данные из открытого датасета **MovieLens**, который собирается исследовательской группой GroupLens в Миннесотском университете. Данные представляют собой историю реальных оценок (от 1 до 5 звезд), которые пользователи выставляли фильмам.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Инициализируем Spark-сессию под единым именем
spark = SparkSession.builder \
    .appName("pyspark_als_demo") \
    .master("local[*]") \
    .getOrCreate()

# Загружаем историю оценок пользователей из локальной папки data/
ratings = spark.read.csv("data/ratings.csv", header=True, inferSchema=True)
ratings.show(5)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows


### Неперсонализированные стратегии

Базовый подход к рекомендации для новых пользователей — выдача популярных объектов. Однако в бизнесе важно рекомендовать не только "всегда популярное" (например, старую классику кино), но и **свежие тренды**. В нашем датасете есть колонка `timestamp` — время, когда пользователь поставил оценку. Давайте найдем фильмы, которые стали популярны в последнее время (в конце истории наблюдений).

Чтобы сформировать топ трендов, мы:
1. Отфильтруем только те оценки, которые были поставлены за последние полгода (180 дней).
2. Сгруппируем данные по фильмам и посчитаем количество оценок и средний балл.
3. **Важное условие стабильности:** мы оставим только те фильмы, которые получили не менее 10 оценок за этот период. Это защитит наш топ от случайных фильмов, которым один человек поставил 5 баллов.

При желании мы могли бы ужесточить фильтр и, например, дополнительно отбросить все фильмы со средним рейтингом ниже 4.0, но в данном примере ограничимся фильтром по количеству голосов.

In [ ]:
# Находим максимальное время в датасете (условное "сегодня")
max_timestamp = ratings.select(F.max("timestamp")).collect()[0][0]

# Вычисляем временную границу за последние полгода (в секундах)
half_year_seconds = 180 * 24 * 60 * 60
border_time = max_timestamp - half_year_seconds

# Фильтруем и агрегируем "свежие тренды"
fresh_trends = ratings.filter(F.col("timestamp") >= border_time) \
    .groupBy("movieId") \
    .agg(
        F.count("rating").alias("recent_reviews_count"),
        F.round(F.avg("rating"), 2).alias("recent_avg_rating")
    ) \
    .filter(F.col("recent_reviews_count") >= 10) \
    .orderBy(F.desc("recent_reviews_count"), F.desc("recent_avg_rating"))

print("Свежие тренды за последние полгода активности датасета (только ID фильмов):")
fresh_trends.show(5)

Свежие тренды за последние полгода активности датасета (только ID фильмов):
+-------+--------------------+-----------------+
|movieId|recent_reviews_count|recent_avg_rating|
+-------+--------------------+-----------------+
|   7153|                  15|             3.97|
|   2571|                  14|             4.29|
|   5952|                  14|             4.14|
|   4993|                  13|             4.15|
| 122912|                  13|              4.0|
+-------+--------------------+-----------------+
only showing top 5 rows


### Обогащение данных

Смотреть на сухие числовые `movieId` неудобно ни аналитикам, ни пользователям. Создадим мини-справочник названий и жанров прямо в памяти кластера, чтобы сымитировать работу с внешним текстовым реестром, и соединим его с нашими трендами с помощью операции `.join()`.

In [ ]:
# Создаем тестовый справочник названий для популярных ID из нашего датасета
# (Spark DataFrame из Python-списка)
local_movies_data = [
    {"movieId": 110, "title": "Braveheart (Храброе сердце)", "genre": "Action|Drama"},
    {"movieId": 260, "title": "Star Wars: Episode IV (Звёздные войны)", "genre": "Sci-Fi"},
    {"movieId": 589, "title": "Terminator 2: Judgment Day (Терминатор 2)", "genre": "Action|Sci-Fi"},
    {"movieId": 2571, "title": "The Matrix (Матрица)", "genre": "Sci-Fi"},
    {"movieId": 356, "title": "Forrest Gump (Форрест Гамп)", "genre": "Comedy|Drama"}
]

# Создаем DataFrame справочника
movies_titles = spark.createDataFrame(local_movies_data)
print("Созданный справочник названий фильмов:")
movies_titles.show(truncate=False)

print("="*50)

# Присоединяем текстовые названия по ключу movieId
# Используем тип 'inner', чтобы оставить только те фильмы, для которых у нас есть названия
enriched_trends = fresh_trends.join(movies_titles, on="movieId", how="inner")

print("Результат обогащения данных:")
enriched_trends.select("title", "genre", "recent_reviews_count", "recent_avg_rating").show(truncate=False)

Созданный справочник названий фильмов:
+-------------+-------+-----------------------------------------+
|genre        |movieId|title                                    |
+-------------+-------+-----------------------------------------+
|Action|Drama |110    |Braveheart (Храброе сердце)              |
|Sci-Fi       |260    |Star Wars: Episode IV (Звёздные войны)   |
|Action|Sci-Fi|589    |Terminator 2: Judgment Day (Терминатор 2)|
|Sci-Fi       |2571   |The Matrix (Матрица)                     |
|Comedy|Drama |356    |Forrest Gump (Форрест Гамп)              |
+-------------+-------+-----------------------------------------+

Результат обогащения данных:
+---------------------------+------------+--------------------+-----------------+
|title                      |genre       |recent_reviews_count|recent_avg_rating|
+---------------------------+------------+--------------------+-----------------+
|The Matrix (Матрица)       |Sci-Fi      |14                  |4.29             |
|Forrest 

### Персонализированный подход. Контентная фильтрация вручную

От работы с неперсонализированными эвристиками мы переходим к алгоритмам персональных рекомендаций. В индустрии их принято разделять на два больших класса:
1. **Контентная фильтрация (Content-Based):** рекомендация объектов, похожих по своим характеристикам на те, которые пользователь уже оценивал положительно (например, совпадение по автору, жанру или издательству).
2. **Коллаборативная фильтрация (Collaborative Filtering):** рекомендация объектов на основе анализа истории взаимодействий похожих пользователей.

Продемонстрируем базовый принцип контентной фильтрации. Допустим, система зафиксировала интерес пользователя к определенному жанру (в нашем примере — "Sci-Fi"). Используя стандартные операции фильтрации и сортировки в Spark DataFrame, мы можем сформировать персональную выборку фильмов, соответствующих данному текстовому признаку.

In [ ]:
# 1. Задаем целевой текстовый признак (жанр) для фильтрации контента
target_genre = "Sci-Fi"

# 2. Рассчитываем базовые статистические показатели для всех фильмов в системе
movie_stats = ratings.groupBy("movieId") \
    .agg(
        F.count("rating").alias("total_votes"),
        F.round(F.avg("rating"), 2).alias("final_score")
    ) \
    .filter(F.col("total_votes") >= 50)  # Исключаем объекты с недостаточным количеством оценок

# 3. Объединяем статистику со справочником и фильтруем по целевому признаку
content_recommendations = movie_stats.join(movies_titles, on="movieId", how="inner") \
    .filter(F.col("genre").like(f"%{target_genre}%")) \
    .orderBy(F.desc("final_score"))

print(f"Результат контентной фильтрации для жанра {target_genre}:")
content_recommendations.select("title", "genre", "final_score").show(truncate=False)

Результат контентной фильтрации для жанра Sci-Fi:
+-----------------------------------------+-------------+-----------+
|title                                    |genre        |final_score|
+-----------------------------------------+-------------+-----------+
|Star Wars: Episode IV (Звёздные войны)   |Sci-Fi       |4.23       |
|The Matrix (Матрица)                     |Sci-Fi       |4.19       |
|Terminator 2: Judgment Day (Терминатор 2)|Action|Sci-Fi|3.97       |
+-----------------------------------------+-------------+-----------+



### Коллаборативная фильтрация и алгоритм ALS

Контентный подход в своем базовом виде работает в рамках явных, заранее известных признаков (жанров, авторов, тегов). Чтобы рекомендовать объекты из смежных категорий контентным методом, необходимо усложнять архитектуру: строить матрицы сходства самих тегов или переводить описания в векторные эмбеддинги, чтобы вычислять семантическую близость между разными жанрами.

Чтобы преодолеть ограниченность признакового пространства, можно использовать коллаборативную фильтрацию. Этот подход анализирует саму матрицу взаимодействий всех пользователей сразу. Если алгоритм обнаруживает устойчивые паттерны совместного потребления (скрытую схожесть вкусов группы людей), система порекомендует объект, даже если у него совершенно другие жанровые теги или незнакомый автор (или вовсе отсутствуют теги).

Однако при переходе к коллаборативному анализу на больших данных классические алгоритмы (например, KNN или расчет попарного косинусного сходства) вызывают ошибку переполнения памяти (OOM) из-за квадратичной вычислительной сложности $O(N^2)$.

Индустриальным стандартом для обхода этого ограничения является **алгоритм  рекомендаций на основе матричных разложений (Alternating Least Squares)**. Он аппроксимирует гигантскую разреженную матрицу оценок, раскладывая её на две плотные матрицы гораздо меньшей размерности: матрицу скрытых факторов пользователей и матрицу скрытых факторов предметов. Каждая итерация ALS оптимизирует эти матрицы поочередно, благодаря чему вычисления эффективно распараллеливаются между узлами Spark-кластера.

Разобьем данные на обучающую и тестовую выборки и обучим модель.

In [ ]:
# Разбиваем данные на train/test
train_data, test_data = ratings.randomSplit([0.8, 0.2], seed=13)

# Инициализируем ALS со стандартными параметрами
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    nonnegative=True
)

# Обучаем модель
model = als.fit(train_data)

# Применяем модель к тестовой выборке
predictions = model.transform(test_data)
predictions.select("userId", "movieId", "rating", "prediction").show(15)

+------+-------+------+----------+
|userId|movieId|rating|prediction|
+------+-------+------+----------+
|     1|      3|   4.0| 3.7956998|
|     1|     70|   3.0|  4.045574|
|     1|    223|   3.0| 4.5615864|
|     1|    423|   3.0|  3.223257|
|     1|    441|   4.0| 4.9689307|
|     1|    480|   4.0| 4.2675705|
|     1|    500|   3.0| 4.0670285|
|     1|    552|   4.0| 3.6677208|
|     1|    661|   5.0|  4.210078|
|     1|    943|   4.0|  4.529901|
|     1|   1136|   5.0|  5.073877|
|     1|   1196|   5.0| 4.9696317|
|     1|   1198|   5.0|  4.689108|
|     1|   1213|   5.0|  4.865066|
|     1|   1240|   5.0|  4.471576|
+------+-------+------+----------+
only showing top 15 rows


### Проблема «Холодного старта»

При применении модели к тестовым данным мы можем столкнуться с ситуацией, когда в столбце `prediction` появляются значения `NaN` (Not a Number). Давайте программно проверим нашу тестовую выборку на наличие таких пропусков и  отфильтруем строки, с которыми алгоритм коллаборативной фильтрации не справился.

In [ ]:
# Считаем количество NaN в столбце prediction
nan_count = predictions.filter(F.isnan(F.col("prediction"))).count()
print(f"{nan_count} (из {predictions.count()})")

predictions.filter(F.isnan(F.col("prediction"))).select("userId", "movieId", "rating", "prediction").show(5)


805 (из 20029)
+------+-------+------+----------+
|userId|movieId|rating|prediction|
+------+-------+------+----------+
|    85|   1140|   5.0|       NaN|
|    85|   1519|   1.0|       NaN|
|   255|   1739|   4.0|       NaN|
|   513|   6145|   2.0|       NaN|
|    34|  26606|   4.5|       NaN|
+------+-------+------+----------+
only showing top 5 rows


### Оценка качества модели

Для оценки качества предсказания используем метрику **RMSE**. Она покажет, на сколько баллов в среднем ошибается наша модель при прогнозировании рейтинга.

Мы можем посчитать ошибку только для тех объектов, про которые одновременно известна и настоящая историческая оценка, и модель смогла предсказать конкретное число. Рассмотрим два способа не учитывать NaN'ы.

In [ ]:
# Настраиваем оценщик метрики RMSE
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

# Способ 1: Очищаем DataFrame вручную, удаляя строки с NaN перед передачей в оценщик
clean_predictions = predictions.filter(~F.isnan(F.col("prediction")))
rmse_clean = evaluator.evaluate(clean_predictions)
print(f"Способ 1. RMSE (после ручной очистки от NaN): {rmse_clean:.4f}")

# Способ 2: Пересобираем модель с параметром coldStartStrategy="drop"
# Этот параметр автоматически велит Спарку исключать NaN при расчете метрик и трансформации
als_with_drop = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True
)
model_drop = als_with_drop.fit(train_data)
predictions_drop = model_drop.transform(test_data)

rmse_drop = evaluator.evaluate(predictions_drop)
print(f"Способ 2. RMSE (через настройку модели coldStartStrategy='drop'): {rmse_drop:.4f}")

Способ 1. RMSE (после ручной очистки от NaN): 0.8758
Способ 2. RMSE (через настройку модели coldStartStrategy='drop'): 0.8758


### Гибридные системы

Параметр `coldStartStrategy="drop"` позволяет корректно рассчитать метрику эффективности на этапе валидации, учитывая только те объекты, для которых модель смогла сформировать прогноз. Однако применение данной стратегии в условиях реальной эксплуатации недопустимо.

Систематическое исключение из рекомендательной выдачи новых или нестандартных профилей пользователей и объектов приводит к критическим дефектам в работе системы:
1. **Снижение метрик удержания (Churn Rate):** Новый или неактивный пользователь при переходе на целевую страницу получит пустой экран либо ошибку интерфейса.
2. **Снижение оборачиваемости ассортимента:** Кинотеатр или стриминговый сервис приобретает права на новые фильмы, но алгоритм машинного обучения не может их порекомендовать (для них нет скрытых факторов). В итоге новый контент остается незамеченным аудиторией.

Для компенсации ограничений матричного разложения архитектуру рекомендательных систем переводят в класс **гибридных систем**, использующих альтернативные алгоритмы при обнаружении значений `NaN`.

#### Стратегии для новых пользователей (Cold Users)
* **Демографическая сегментация:** При отсутствии истории взаимодействий система выполняет перенаправление на неперсонализированный топ популярных объектов, рассчитанный для конкретной демографической группы (на основе возраста, географического положения или пола).
* **Контекстная фильтрация:** Если пользователь совершает переход по конкретному объекту (например, открыл карточку фильма жанра "Научная фантастика"), система формирует список рекомендаций на основе схожести характеристик (Content-Based).

#### Стратегии для новых объектов (Cold Items)
* **Контентное проталкивание:** Система идентифицирует пользователей, в чьих профилях преобладают скрытые факторы, схожие с атрибутами новинки (например, по режиссеру, актерам или жанру), и принудительно внедряет новый объект в их персональную выдачу.
* **Методы направленного исследования:** Выделяется фиксированная доля поискового трафика (например, 5%), на которой пользователям демонстрируются новые объекты в случайном или полуслучайном порядке. Данная стратегия необходима для накопления первичной статистики взаимодействий. Как только объект набирает критическую массу оценок, при следующем итерационном переобучении модели ALS для него успешно вычисляются скрытые факторы, и он интегрируется в общую рекомендательную сеть.